# EDA · entender la fuga en Wong

Cubre solo lo imprescindible para *mirar* la base antes de modelar:

1. **Cargar y apilar** los cinco parquet
2. **Muestrear clientes** (no filas) y comprobar que la muestra representa
3. **Construir el RFM** — `x`, `t_x`, `T`, `monetario` — con sus invariantes
4. **NBD**: ajustar la mezcla Poisson-Gamma con `scipy` y graficarla
5. **BG**: ajustar la mezcla Beta-Geométrica con `scipy` y graficarla

Los pasos 4 y 5 son **las dos mitades del BG/NBD por separado**: el NBD contesta
*«¿a qué ritmo compran los que están vivos?»* y el BG contesta *«¿cuántas compras
aguantan antes de irse?»*. Verlas sueltas —antes de juntarlas en el modelo— es lo
que permite entender de dónde sale la fuga.

Nada de librerías BTYD: todo con `scipy.optimize` y `scipy.special`.

**Decisiones ya tomadas** (ver `Plan de analisis`): corte `2025-05-10` (desde el
11-may-2025 los domingos no cargan), `sk_cliente > 0`, unidad monetaria `margen`.

In [ ]:
from pathlib import Path
from datetime import date
import glob, os

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

RAIZ = Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent

PATRON = str(RAIZ / "raw" / "wong_dia_*.parquet")
CORTE  = date(2025, 5, 10)     # ultimo dia integro: despues fallan los domingos
SEMILLA = 42
N_MUESTRA = 100_000

print(RAIZ)
print(PATRON)

In [ ]:
# ---- estilo de las figuras (paleta categorica fija, se asigna por orden) ----
AZUL, NARANJA, AGUA = "#2a78d6", "#eb6834", "#1baf7a"
TINTA, TINTA2, GRIS = "#0b0b0b", "#52514e", "#c9c8c3"

plt.rcParams.update({
    "figure.figsize": (9, 4.6), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.spines.left": False,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA2,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlecolor": TINTA,
    "axes.grid": True, "grid.color": GRIS, "grid.linewidth": 0.6, "grid.alpha": 0.5,
    "axes.axisbelow": True,
    "xtick.color": TINTA2, "ytick.color": TINTA2,
    "xtick.bottom": True, "ytick.left": False,
    "legend.frameon": False, "font.size": 10,
})

def miles(ax):
    """Separador de miles en el eje y: 15000 se lee peor que 15,000."""
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))

---

## 1 · Cargar y apilar los cinco parquet

Son cinco mediciones **de lo mismo** (cliente-día) en años distintos: se **apilan**
(`concat`), no se cruzan. Si se cruzaran por `sk_cliente`, un cliente con 50 compras
en 2023 y 40 en 2024 generaría 2 000 filas — producto cartesiano, y sin mensaje de error.

Antes de apilar hay que comprobar que los esquemas coinciden. Eso se hace **sin leer
los datos**: `scan_parquet` solo mira la cabecera.

In [ ]:
archivos = sorted(glob.glob(PATRON))
esquemas = []

for f in archivos:
    lf = pl.scan_parquet(f)
    esquemas.append(lf.collect_schema())
    r = lf.select(
        pl.len().alias("filas"),
        pl.col("cod_dia").min().alias("min"),
        pl.col("cod_dia").max().alias("max"),
    ).collect()
    print(f"{os.path.basename(f):<24} {r['filas'][0]:>12,}  {r['min'][0]} -> {r['max'][0]}")

print("\nesquemas identicos:", all(e == esquemas[0] for e in esquemas))
print(esquemas[0])

`scan_parquet` acepta el patrón con `*`: los cinco se abren de una vez y polars los
apila solo. La comprobación que importa es que **la suma de las filas de los cinco
archivos por separado dé exactamente el total de la tabla junta**.

In [ ]:
todo = pl.scan_parquet(PATRON)

r = todo.select(
    pl.len().alias("filas"),
    pl.col("sk_cliente").n_unique().alias("clientes"),
    pl.col("cod_dia").min().alias("dmin"),
    pl.col("cod_dia").max().alias("dmax"),
).collect()

suma = sum(pl.scan_parquet(f).select(pl.len()).collect().item() for f in archivos)

print(f"junto: {r['filas'][0]:,} filas | {r['clientes'][0]:,} clientes | "
      f"{r['dmin'][0]} -> {r['dmax'][0]}")
print(f"suma por separado: {suma:,}  | coincide: {suma == r['filas'][0]}")

**Filas se suman; clientes no.** El mismo cliente reaparece año tras año, así que
`n_unique` sobre la tabla junta es mucho menor que la suma de los `n_unique` anuales.
Es la primera vez que aparece una operación **no aditiva**, y conviene fijarla: sumar
ventas por año está bien, sumar clientes por año no.

In [ ]:
por_anio = [pl.scan_parquet(f).select(pl.col("sk_cliente").n_unique()).collect().item()
            for f in archivos]

for f, n in zip(archivos, por_anio):
    print(f"{os.path.basename(f)[9:13]}: {n:>9,}")
print(f"\nsuma por anio: {sum(por_anio):,}")
print(f"real junto:    {r['clientes'][0]:,}   <- {sum(por_anio) - r['clientes'][0]:,} menos")

### La carga de verdad

Perezosa hasta el final: `scan` → filtros → columnas → `collect()`. Los filtros y la
selección se aplican **mientras lee**, así que nunca entra en memoria lo que no se usa
(≈540 MB frente a ≈2 200 MB de pico con `read_parquet`).

- `sk_cliente > 0` — el `0` es el agregado de venta no identificada, no una persona
- `cod_dia` (entero `20240315`) → fecha de verdad, en dos saltos: entero → texto → fecha
- todo lo posterior al `CORTE` se descarta

In [ ]:
datos = (
    pl.scan_parquet(PATRON)
      .filter(pl.col("sk_cliente") > 0)
      .with_columns(
          pl.col("cod_dia").cast(pl.Utf8).str.to_date("%Y%m%d").alias("fecha")
      )
      .filter(pl.col("fecha") <= CORTE)
      .select("sk_cliente", "fecha", "margen")
      .collect()
)

print(f"{len(datos):,} filas | {datos['sk_cliente'].n_unique():,} clientes")
print(datos["fecha"].min(), "->", datos["fecha"].max())
datos.head()

---

## 2 · La muestra: clientes, nunca filas

**La unidad de muestreo tiene que ser la unidad de análisis.** Analizamos clientes,
así que se muestrean clientes y se trae **toda** su historia.

Muestrear filas rompe al cliente: su frecuencia deja de ser real, su última compra ya
no es su última compra, y la recencia queda inventada. Los tres números sobre los que
se sostiene todo el modelo saldrían falsos.

Tres movimientos: **lista de clientes únicos → elegir N al azar → traer su historia completa.**

> **`seed` no basta.** `.unique()` no garantiza el orden (polars lo resuelve en
> paralelo), y `sample(seed=42)` sobre una lista barajada distinta elige *clientes*
> distintos. Reproducible = **semilla fija _y_ orden determinista**: por eso el `.sort()`.

In [ ]:
ids = (datos.select("sk_cliente")
            .unique()
            .sort("sk_cliente")          # sin esto NO es reproducible
            .sample(N_MUESTRA, seed=SEMILLA))

muestra = datos.join(ids, on="sk_cliente", how="inner")

print(f"poblacion: {len(datos):>12,} filas | {datos['sk_cliente'].n_unique():>9,} clientes")
print(f"muestra:   {len(muestra):>12,} filas | {muestra['sk_cliente'].n_unique():>9,} clientes")
print(f"\nsuma de ids (control de reproducibilidad): {ids['sk_cliente'].sum():,}")

### ¿Se parece a la población?

Filas por cliente es el chequeo rápido. El honesto compara **varias estadísticas a la
vez**, porque una muestra puede acertar la media y fallar la forma. Que coincidan
media *y* mediana *y* la proporción de clase cero es lo que de verdad tranquiliza.

In [ ]:
def perfil(df):
    t = (df.group_by("sk_cliente")
           .agg(pl.col("fecha").min().alias("p"),
                pl.col("fecha").max().alias("u"),
                pl.len().alias("v"))
           .with_columns(
               (pl.col("v") - 1).alias("x"),
               (pl.lit(CORTE) - pl.col("p")).dt.total_days().alias("T")))
    return {
        "filas":     len(df),
        "clientes":  len(t),
        "filas/cli": len(df) / len(t),
        "x medio":   t["x"].mean(),
        "x mediana": t["x"].median(),
        "% x=0":     100 * (t["x"] == 0).sum() / len(t),
        "T medio":   t["T"].mean(),
    }

a, b_ = perfil(datos), perfil(muestra)
for k in a:
    print(f"{k:>10} | poblacion {a[k]:>13,.2f} | muestra {b_[k]:>12,.2f}")

---

## 3 · La tabla RFM

De ~1 millón de filas a una fila por cliente. El modelo necesita exactamente tres
números — y en la teoría se demuestra que $(x, t_x, T)$ es **estadístico suficiente**:
al agregar no se pierde nada, las fechas intermedias se cancelan solas.

| | qué es | cómo se calcula |
|---|---|---|
| `x` | cuántas veces **volvió** | días distintos de compra **menos 1** |
| `t_x` | cuánto duró su vida observada | días entre la primera y la última compra |
| `T` | cuánto tiempo lo hemos mirado | días entre la primera compra y el corte |

**El `-1` de `x` no es un detalle.** La primera compra es el nacimiento del cliente,
la razón por la que aparece en la base. Si contara, todos tendrían al menos 1 por
construcción y ese 1 no informaría de nada. Con `x = recompras`, el `0` significa algo
real y observable: *«compró una vez y no volvió»*.

Y la comparación que lo decide todo: **si `t_x` está muy por debajo de `T`, ese cliente
lleva mucho tiempo callado.**

Para el monetario, dos trampas conocidas (las dos costaron caro en METRO):
`margen_repeat` **excluye la primera compra** —Gamma-Gamma modela el gasto de las
recompras, y la primera suele ser atípica— y el ticket de quien nunca recompró es
`None`, **nunca `0`**: no es un ticket de cero, es un ticket *desconocido*.

In [ ]:
tabla = (
    muestra.group_by("sk_cliente")
           .agg(
               pl.col("fecha").min().alias("primera"),
               pl.col("fecha").max().alias("ultima"),
               pl.len().alias("visitas"),
               pl.col("margen")
                 .filter(pl.col("fecha") > pl.col("fecha").min())
                 .sum().alias("margen_repeat"),
           )
           .with_columns(
               (pl.col("visitas") - 1).alias("x"),
               (pl.col("ultima") - pl.col("primera")).dt.total_days().alias("t_x"),
               (pl.lit(CORTE) - pl.col("primera")).dt.total_days().alias("T"),
           )
           .with_columns(
               pl.when(pl.col("x") > 0)
                 .then(pl.col("margen_repeat") / pl.col("x"))
                 .otherwise(None)              # None, NUNCA 0
                 .alias("monetario"),
           )
           .drop("visitas", "margen_repeat")
           .sort("sk_cliente")
)

tabla.head()

### Las invariantes

Cosas que **por definición** tienen que cumplirse. Si una falla, hay un bug —
garantizado. Escribirlas antes de mirar los resultados es la costumbre más rentable
de todo el análisis.

1. **Una fila por cliente.** Si salen más, el `group_by` no agrupó por lo que crees.
2. **`x` nunca negativo.** Todo cliente tiene al menos una visita.
3. **`t_x` nunca mayor que `T`.** La última compra no puede ser posterior al corte;
   si falla, hay fechas de dos periodos mezcladas.
4. **`x == 0` si y solo si `t_x == 0`.** Doble implicación, y por eso la más potente:
   comprueba `x` y `t_x` a la vez. Como el grano es cliente-día, no puede haber dos
   compras el mismo día.
5. **`monetario` nulo ⇔ `x == 0`.** Confirma que el `when/then/otherwise` quedó bien.

In [ ]:
print("filas == clientes  :", len(tabla) == muestra["sk_cliente"].n_unique())
print("x < 0              :", (tabla["x"] < 0).sum(), " <- debe ser 0")
print("t_x > T            :", (tabla["t_x"] > tabla["T"]).sum(), " <- debe ser 0")
print("x==0 <=> t_x==0    :", ((tabla["x"] == 0) == (tabla["t_x"] == 0)).all())
print("nulos == (x==0)    :", tabla["monetario"].null_count() == (tabla["x"] == 0).sum(),
      f"  ({tabla['monetario'].null_count():,})")
print()
for c in ("x", "t_x", "T"):
    print(f"{c:<5}: media {tabla[c].mean():7.2f}   mediana {tabla[c].median():6.0f}   max {tabla[c].max()}")
print(f"\nmonetario negativo: {(tabla['monetario'] < 0).sum():,} clientes"
      f"   <- Gamma-Gamma no los admite, hay que excluirlos en el CLV")

In [ ]:
tabla.select("x", "t_x", "T", "monetario").describe()

### La forma de `x`

Primer gráfico y primer diagnóstico. Lo que hay que mirar es la distancia entre media
y mediana: si son muy distintas, la población es brutalmente heterogénea — y esa
heterogeneidad es exactamente lo que la Gamma del modelo existe para representar.

In [ ]:
xs = tabla["x"].to_numpy()
TOPE = 30
recorte = np.clip(xs, 0, TOPE)

fig, ax = plt.subplots()
ax.hist(recorte, bins=np.arange(-0.5, TOPE + 1.5), color=AZUL, rwidth=0.86)
ax.axvline(np.median(xs), color=NARANJA, lw=2, ls="--")
ax.axvline(min(np.mean(xs), TOPE), color=TINTA, lw=2, ls=":")
ax.annotate(f"mediana {np.median(xs):.0f}", (np.median(xs), 0), xytext=(6, 190),
            textcoords="offset points", color=NARANJA, fontweight="bold")
ax.annotate(f"media {np.mean(xs):.1f}", (np.mean(xs), 0), xytext=(6, 150),
            textcoords="offset points", color=TINTA, fontweight="bold")
ax.set_title(f"Recompras por cliente (n = {len(xs):,}; la ultima barra agrupa {TOPE}+)")
ax.set_xlabel("x  ·  recompras")
ax.set_ylabel("clientes")
miles(ax)
plt.tight_layout(); plt.show()

p0 = 100 * (xs == 0).mean()
print(f"clase cero (x = 0): {(xs == 0).sum():,} clientes  ({p0:.1f}%)")
print(f"media {xs.mean():.2f}  |  mediana {np.median(xs):.0f}  |  p90 {np.percentile(xs, 90):.0f}  |  max {xs.max()}")

Y el otro rasgo que condiciona todo lo que viene: **la distribución de `T`**. Si hay
un pico en el máximo, no es una cohorte real de captación — es el **borde de la tabla**:
todos los que ya eran clientes cuando empiezan los datos aparecen de golpe el primer día.

In [ ]:
fig, ax = plt.subplots()
ax.hist(tabla["T"].to_numpy(), bins=60, color=AZUL)
ax.set_title("Antiguedad observada T (dias desde la primera compra hasta el corte)")
ax.set_xlabel("T  ·  dias")
ax.set_ylabel("clientes")
miles(ax)
plt.tight_layout(); plt.show()

Tv = tabla["T"].to_numpy()
print(f"T: media {Tv.mean():.1f}  mediana {np.median(Tv):.0f}  max {Tv.max()}")
print(f"clientes con T >= {Tv.max() - 31}: {(Tv >= Tv.max() - 31).sum():,} "
      f"({100 * (Tv >= Tv.max() - 31).mean():.1f}%)  <- el borde de dic-2022")

---

## 4 · NBD — el ritmo de compra, con `scipy`

La primera mitad del BG/NBD. Contesta **«¿a qué ritmo compra la gente?»** e ignora por
completo el abandono: aquí nadie se muere, todos siguen comprando para siempre a su
propio ritmo.

**Dos ladrillos:**

- **Poisson**, el conteo de *un* individuo: si compra a tasa $\lambda$ durante un
  tiempo $T$, $X \mid \lambda \sim \text{Poisson}(\lambda T)$. Implica
  $E[X] = \text{Var}(X)$ — y **esa igualdad es la que falla en la práctica**.
- **Gamma**, el reparto de las tasas entre clientes:
  $f(\lambda) = \frac{\alpha^{r}}{\Gamma(r)}\lambda^{r-1}e^{-\alpha\lambda}$,
  con $E[\lambda] = r/\alpha$. Se elige porque es positiva, flexible y —lo decisivo—
  **conjugada del Poisson**, así que la mezcla tiene solución cerrada.

**La mezcla** da la binomial negativa:

$$P(X = x) = \frac{\Gamma(r+x)}{\Gamma(r)\,x!}\left(\frac{\alpha}{\alpha+T}\right)^{r}\left(\frac{T}{\alpha+T}\right)^{x}, \qquad p = \frac{\alpha}{\alpha+T}$$

$$E[X] = \frac{rT}{\alpha}, \qquad \text{Var}(X) = \underbrace{\frac{rT}{\alpha}}_{\text{azar del conteo}} + \underbrace{\frac{rT^{2}}{\alpha^{2}}}_{\text{heterogeneidad}}$$

> **$p$ depende del individuo.** $p = \alpha/(\alpha+T_i)$ es función del tiempo que se
> observó a *ese* cliente. Aquí las historias tienen longitudes muy distintas, así que
> hay **un $p$ por cliente**. Ignorarlo obliga al modelo a explicar con un único $p$ a
> alguien observado 20 días y a alguien observado 890 — es la causa más frecuente de que
> el ajuste no converja.

### Antes de ajustar: la sobredispersión

Si $\text{Var}/E$ sale ≈ 1, un Poisson simple bastaría y la Gamma sobra. Cuanto más
alto, más heterogénea es la población.

In [ ]:
from scipy.special import gammaln, betaln
from scipy.optimize import minimize
from scipy.stats import nbinom, poisson, gamma as gamma_dist, beta as beta_dist

x_obs = tabla["x"].to_numpy().astype(float)
T_sem = tabla["T"].to_numpy().astype(float) / 7.0     # semanas: lambda por semana

print(f"media    {x_obs.mean():8.3f}")
print(f"varianza {x_obs.var():8.3f}")
print(f"var/media {x_obs.var() / x_obs.mean():7.2f}   <- 1.0 seria Poisson puro")

### El ajuste

Máxima verosimilitud con `scipy.optimize.minimize`, con **los dos trucos** de siempre:

1. **Se optimiza sobre logaritmos.** $r$ y $\alpha$ tienen que ser positivos. En vez de
   restringir al optimizador (frágil), se le pasa $\ln r, \ln\alpha$ y dentro se hace
   `np.exp()`: se mueve por todos los reales y al exponenciar siempre sale positivo.
2. **Todo en espacio logarítmico**, con `gammaln`. La verosimilitud es un producto de
   50 000 números pequeños; calculada directa se va a cero y el logaritmo explota.

$$\ln L = \sum_i \left[\ln\Gamma(r+x_i) - \ln\Gamma(r) - \ln\Gamma(x_i+1) + r\ln\frac{\alpha}{\alpha+T_i} + x_i\ln\frac{T_i}{\alpha+T_i}\right]$$

In [ ]:
def nll_nbd(log_par, x, T):
    "-log verosimilitud de la NBD con un p por cliente. log_par = log(r, alpha)."
    r, alpha = np.exp(log_par)
    ln_p = r * (np.log(alpha) - np.log(alpha + T))
    # x=0 no aporta al segundo termino; T=0 solo ocurre si x=0, asi que 0*log(0) se anula
    ln_q = np.where(x > 0, x * (np.log(np.maximum(T, 1e-12)) - np.log(alpha + T)), 0.0)
    ll = gammaln(r + x) - gammaln(r) - gammaln(x + 1) + ln_p + ln_q
    return -ll.sum()


res = minimize(nll_nbd, np.log([1.0, 1.0]), args=(x_obs, T_sem), method="Nelder-Mead",
               options={"maxiter": 10_000, "xatol": 1e-8, "fatol": 1e-8})
res = minimize(nll_nbd, res.x, args=(x_obs, T_sem), method="Nelder-Mead")   # 2a pasada
r_nbd, alpha_nbd = np.exp(res.x)

print(f"convergio: {res.success}   LL = {-res.fun:,.1f}")
print(f"r     = {r_nbd:8.4f}")
print(f"alpha = {alpha_nbd:8.4f}")
print(f"\nE[lambda] = r/alpha = {r_nbd / alpha_nbd:.4f} compras/semana"
      f"  ({r_nbd / alpha_nbd * 52:.1f} al ano)")
print(f"Var[lambda] = r/alpha^2 = {r_nbd / alpha_nbd**2:.4f}")

**Cómo se lee $r$:** $r > 1$ es una campana asimétrica (población homogénea); $r = 1$ es
exponencial; $r < 1$ da una densidad en forma de **«L»**, infinita cerca de cero y con
cola larga — mucha masa de clientes casi inactivos y una minoría intensa. **Cuanto menor
es $r$, mayor la heterogeneidad.**

In [ ]:
# la Gamma ajustada contra las tasas empiricas x_i / T_i
vivos = T_sem > 0
lam_emp = x_obs[vivos] / T_sem[vivos]

fig, ax = plt.subplots()
tope = np.percentile(lam_emp, 99)
alturas, _, _ = ax.hist(lam_emp, bins=np.linspace(0, tope, 70), density=True,
                        color=AZUL, alpha=0.85, label="observado  $x_i / T_i$")
rej = np.linspace(1e-4, tope, 400)
ax.plot(rej, gamma_dist.pdf(rej, r_nbd, scale=1 / alpha_nbd),
        color=NARANJA, lw=2, label=f"Gamma ajustada  (r={r_nbd:.3f}, $\\alpha$={alpha_nbd:.3f})")
# con r < 1 la densidad diverge en 0: se recorta el eje o no se ve nada mas
ax.set_ylim(0, alturas.max() * 1.3)
ax.annotate("con r < 1 la Gamma\ndiverge en 0 (eje recortado)", (0.02, alturas.max() * 1.3),
            xytext=(30, -18), textcoords="offset points", color=TINTA2, fontsize=9)
ax.set_title("Heterogeneidad del ritmo de compra")
ax.set_xlabel("$\\lambda$  ·  compras por semana")
ax.set_ylabel("densidad")
ax.legend()
plt.tight_layout(); plt.show()

q = gamma_dist.ppf([.25, .50, .75, .90], r_nbd, scale=1 / alpha_nbd)
print("percentiles de lambda (compras/semana):",
      "  ".join(f"p{p}={v:.3f}" for p, v in zip([25, 50, 75, 90], q)))
print(f"media {r_nbd / alpha_nbd:.3f}  |  mediana {q[1]:.3f}"
      "   <- si la media dobla a la mediana, el 'cliente promedio' no representa a nadie")

### La primera prueba de todo modelo: predicho vs. real

Cuántos clientes con 0, 1, 2, … recompras. Si el modelo falla aquí, nada de lo demás
importa. Se compara la NBD contra un **Poisson único** (un solo $\lambda$ para toda la
base) para ver qué aporta la heterogeneidad.

La predicción no es una pmf sola: cada cliente tiene su propio $p_i = \alpha/(\alpha+T_i)$,
así que la frecuencia esperada es el **promedio de las pmf individuales**.

In [ ]:
K = 8                                    # 0..7 y luego "8+"
p_i  = alpha_nbd / (alpha_nbd + T_sem)
lam_pois = x_obs.sum() / T_sem.sum()     # MLE del Poisson unico
mu_i = lam_pois * T_sem

ks = np.arange(K)
obs  = np.array([(x_obs == k).sum() for k in ks] + [(x_obs >= K).sum()], float)
nbd  = np.array([nbinom.pmf(k, r_nbd, p_i).mean() for k in ks]
                + [nbinom.sf(K - 1, r_nbd, p_i).mean()]) * len(x_obs)
pois = np.array([poisson.pmf(k, mu_i).mean() for k in ks]
                + [poisson.sf(K - 1, mu_i).mean()]) * len(x_obs)

etiq = [str(k) for k in ks] + [f"{K}+"]
pos  = np.arange(K + 1)

fig, ax = plt.subplots()
h_obs = ax.bar(pos, obs, width=0.72, color=AZUL, label="observado")
h_nbd, = ax.plot(pos, nbd,  "o-", color=NARANJA, lw=2, ms=8, mec="white", mew=1.5,
                 label="NBD (Poisson-Gamma)")
h_poi, = ax.plot(pos, pois, "s-", color=AGUA, lw=2, ms=8, mec="white", mew=1.5,
                 label=f"Poisson unico ($\\lambda$={lam_pois:.3f})")

# el 8+ del Poisson se sale de la escala; recortar el eje deja ver el detalle de 0..7
techo = obs.max() * 1.18
ax.set_ylim(0, techo)
if pois[-1] > techo:
    ax.annotate(f"Poisson: {pois[-1]:,.0f}\n(fuera de escala)", (pos[-1], techo),
                xytext=(-92, -34), textcoords="offset points", color=AGUA,
                fontsize=9, fontweight="bold")

ax.set_xticks(pos); ax.set_xticklabels(etiq)
ax.set_title("Distribucion de recompras: observado vs. modelos")
ax.set_xlabel("x  ·  recompras en la ventana observada")
ax.set_ylabel("clientes")
miles(ax); ax.legend(handles=[h_obs, h_nbd, h_poi])
plt.tight_layout(); plt.show()

print(f"{'x':>3} {'observado':>11} {'NBD':>11} {'Poisson':>11}   {'error NBD':>10}")
for e, o, n, p in zip(etiq, obs, nbd, pois):
    print(f"{e:>3} {o:>11,.0f} {n:>11,.0f} {p:>11,.0f}   {100*(n-o)/max(o,1):>9.1f}%")
print(f"\nchi2 NBD     : {(((obs - nbd)**2) / nbd).sum():>12,.0f}")
print(f"chi2 Poisson : {(((obs - pois)**2) / pois).sum():>12,.0f}")

Dos lecturas del gráfico:

**La heterogeneidad no es un adorno.** El Poisson único, con el mismo dato, coloca a
casi todo el mundo en el montón de «8 o más» y deja el 0 casi vacío: predice ~700 clientes
de una sola compra cuando hay 15 158. La diferencia de $\chi^2$ entre ambos es de tres
órdenes de magnitud. **Var/media ≈ 111** ya lo anunciaba: con esa sobredispersión, un solo
$\lambda$ no puede describir a nadie.

**Dónde falla el NBD.** El ajuste es bueno, pero el error no está repartido al azar:
se queda **corto en x = 1, 2 y 3** (−19 %, −13 %, −11 %) y **se pasa en 8+** (+8 %). Es
decir, hay más gente de la que el modelo espera que compró unas pocas veces y paró, y
menos de la que espera entre los muy frecuentes. Ese patrón tiene nombre: son los que
**se fueron**, y un modelo donde nadie muere no tiene forma de acomodarlos salvo torciendo
la Gamma.

> **Lo que el NBD no puede ver.** Ajusta *cuántas* compras hace cada cliente, pero no
> *cuándo* fue la última: a un cliente con 30 recompras que lleva un año callado le asigna
> el mismo ritmo que a uno igual de frecuente que compró ayer. `t_x` no entra en la
> verosimilitud. Toda la información de fuga está justamente ahí — y es lo que añade
> la segunda mitad del modelo.

---

## 5 · BG — el abandono, con `scipy`

La segunda mitad. Contesta **«¿cuántas compras aguanta un cliente antes de irse?»**

**El mecanismo:** después de *cada* transacción el cliente lanza una moneda y se vuelve
inactivo con probabilidad $p$. El número de transacciones hasta abandonar es una
**geométrica desplazada**: $P(J = j) = p(1-p)^{j-1}$. Y la $p$ no es igual para todos —
se reparte entre clientes según una **Beta**:

$$f(p \mid a, b) = \frac{p^{a-1}(1-p)^{b-1}}{B(a,b)}, \qquad E[p] = \frac{a}{a+b}$$

Al mezclar geométrica y Beta sale la **beta-geométrica**:

$$P(J = j) = \frac{B(a+1,\; b+j-1)}{B(a,b)}, \qquad
P(J > j) = \frac{B(a,\; b+j)}{B(a,b)}$$

**Cómo se leen $a$ y $b$:** $a$ empuja hacia clientes **volubles**, $b$ hacia clientes
**fieles**. Con $a<1$ y $b>1$ la densidad tiene forma de **«J» decreciente** (mayoría
fiel, minoría muy volátil); con $a<1$ y $b<1$ tiene forma de **«U»** (población
polarizada en los dos extremos).

> **La consecuencia que más cuesta creer:** en BG/NBD la vida útil resulta exponencial
> de tasa $\lambda p$ — atada al ritmo de compra. Quien compra el doble de rápido lanza
> la moneda el doble de veces y, en calendario, **muere el doble de rápido**. De ahí sale
> que $E[X(\infty) \mid \lambda, p] = 1/p$: **el total de compras de por vida depende solo
> de $p$**; $\lambda$ solo decide a qué velocidad se consumen.

### El ajuste, y su supuesto incómodo

Para ajustar el BG **suelto** hace falta observar $j$, el número de transacciones hasta
el abandono. No se observa: los vivos siguen comprando. La aproximación de EDA es
suponer que **todos abandonaron justo después de su última compra**, o sea
$j = x + 1$.

Ese supuesto **es exactamente lo que el BG/NBD completo existe para evitar** — allí
$t_x$ y $T$ separan al muerto del lento. Aquí sirve para ver la *forma* del abandono,
sabiendo que **sobreestima $p$**: censura tratada como muerte.

In [ ]:
j_obs = x_obs + 1.0        # transacciones hasta el abandono (censura tratada como muerte)

def nll_bg(log_par, j):
    "-log verosimilitud de la beta-geometrica desplazada. log_par = log(a, b)."
    a, b = np.exp(log_par)
    return -np.sum(betaln(a + 1.0, b + j - 1.0) - betaln(a, b))


res_bg = minimize(nll_bg, np.log([1.0, 1.0]), args=(j_obs,), method="Nelder-Mead",
                  options={"maxiter": 10_000, "xatol": 1e-8, "fatol": 1e-8})
res_bg = minimize(nll_bg, res_bg.x, args=(j_obs,), method="Nelder-Mead")
a_bg, b_bg = np.exp(res_bg.x)

print(f"convergio: {res_bg.success}   LL = {-res_bg.fun:,.1f}")
print(f"a = {a_bg:8.4f}")
print(f"b = {b_bg:8.4f}")
print(f"\nE[p] = a/(a+b) = {a_bg / (a_bg + b_bg):.4f}   <- prob. media de abandonar tras una compra")

forma = ("'J' decreciente: mayoria fiel, minoria muy volatil" if a_bg < 1 <= b_bg else
         "'U': poblacion polarizada en los dos extremos"      if a_bg < 1 and b_bg < 1 else
         "campana: poblacion relativamente homogenea")
print(f"forma de la Beta (a={a_bg:.3f}, b={b_bg:.3f}): {forma}")

In [ ]:
fig, ax = plt.subplots()
rej_p = np.linspace(1e-4, 1 - 1e-4, 500)
dens = beta_dist.pdf(rej_p, a_bg, b_bg)
ax.plot(rej_p, dens, color=NARANJA, lw=2)
ax.fill_between(rej_p, dens, color=NARANJA, alpha=0.12)
ax.axvline(a_bg / (a_bg + b_bg), color=TINTA, lw=2, ls=":")
ax.annotate(f"E[p] = {a_bg/(a_bg+b_bg):.3f}", (a_bg / (a_bg + b_bg), 0),
            xytext=(8, 40), textcoords="offset points", color=TINTA, fontweight="bold")
ax.set_ylim(0, np.percentile(dens, 97) * 1.6)
ax.set_title("Heterogeneidad del abandono: densidad Beta ajustada")
ax.set_xlabel("p  ·  probabilidad de abandonar despues de una compra")
ax.set_ylabel("densidad")
plt.tight_layout(); plt.show()

qp = beta_dist.ppf([.25, .50, .75, .90], a_bg, b_bg)
print("percentiles de p:", "  ".join(f"p{k}={v:.3f}" for k, v in zip([25, 50, 75, 90], qp)))

### La curva de retención — el gráfico de la fuga

De los clientes que llegaron a hacer $j$ recompras, ¿qué fracción hizo la siguiente?

$$\text{retencion}(j) = \frac{\#\{x \ge j+1\}}{\#\{x \ge j\}}
\qquad\text{y el BG predice}\qquad \frac{b+j}{a+b+j}$$

Lo que hay que mirar es que la curva **sube**. No es que los clientes se vuelvan más
fieles con el tiempo: es que **la mezcla se va limpiando sola**. Los volubles (con $p$
alta) se caen primero, así que quien sobrevive a 10 recompras es, por selección, de los
que tenían $p$ baja. Una geométrica pura daría una recta horizontal — toda la curvatura
es heterogeneidad.

In [ ]:
JMAX = 20
xi = x_obs.astype(int)
s  = np.array([(xi >= k).sum() for k in range(JMAX + 2)], float)

ret_obs = s[1:JMAX + 1] / np.maximum(s[:JMAX], 1)
jj      = np.arange(JMAX)
ret_bg  = (b_bg + jj) / (a_bg + b_bg + jj)

fig, ax = plt.subplots()
ax.plot(jj, ret_obs, "o-", color=AZUL, lw=2, ms=8, mec="white", mew=1.5,
        label="observado")
ax.plot(jj, ret_bg,  "-",  color=NARANJA, lw=2,
        label=f"beta-geometrica ajustada (a={a_bg:.3f}, b={b_bg:.3f})")
ax.set_ylim(0, 1.02)
ax.set_title("Retencion por numero de recompras acumuladas")
ax.set_xlabel("j  ·  recompras ya hechas")
ax.set_ylabel("P(hace la recompra j+1  |  hizo j)")
ax.set_xticks(jj[::2])
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

print(f"{'j':>3} {'sobreviven':>11} {'ret. obs':>10} {'ret. BG':>9}")
for k in list(range(6)) + [8, 10, 15, 19]:
    print(f"{k:>3} {s[k]:>11,.0f} {ret_obs[k]:>10.3f} {ret_bg[k]:>9.3f}")

La curva observada arranca **por debajo** de la ajustada (0,697 contra 0,734) y luego va
por encima: el BG no consigue reproducir a la vez el salto del primer paso —la clase cero—
y la meseta posterior. Es el mismo defecto que se le conoce al BG/NBD, y la razón de ser
del MBG/NBD.

Y la misma información vista como distribución: cuántos clientes se quedan en 0, 1, 2, …
recompras, observado frente a lo que predice la beta-geométrica ajustada.

In [ ]:
bg_pred = np.array([np.exp(betaln(a_bg + 1, b_bg + k) - betaln(a_bg, b_bg)) for k in ks]
                   + [np.exp(betaln(a_bg, b_bg + K) - betaln(a_bg, b_bg))]) * len(x_obs)

fig, ax = plt.subplots()
h_obs = ax.bar(pos, obs, width=0.72, color=AZUL, label="observado")
h_bg, = ax.plot(pos, bg_pred, "o-", color=NARANJA, lw=2, ms=8, mec="white", mew=1.5,
                label="beta-geometrica ajustada")
ax.legend(handles=[h_obs, h_bg])
ax.set_xticks(pos); ax.set_xticklabels(etiq)
ax.set_title("Recompras hasta el abandono: observado vs. BG")
ax.set_xlabel("x  ·  recompras")
ax.set_ylabel("clientes")
miles(ax)
plt.tight_layout(); plt.show()

### Cuánto depende $p$ de a quién mires

El mismo ajuste, restringido al cuartil de clientes más antiguos. Si el número fuera
una propiedad estable de la base, debería moverse poco.

In [ ]:
UMBRAL = int(np.percentile(tabla["T"].to_numpy(), 75))
vet = tabla["T"].to_numpy() >= UMBRAL
j_vet = x_obs[vet] + 1.0

rv = minimize(nll_bg, np.log([1.0, 1.0]), args=(j_vet,), method="Nelder-Mead",
              options={"maxiter": 10_000})
rv = minimize(nll_bg, rv.x, args=(j_vet,), method="Nelder-Mead")
a_v, b_v = np.exp(rv.x)

print(f"toda la muestra (n={len(j_obs):,}):        a={a_bg:.4f}  b={b_bg:.4f}  E[p]={a_bg/(a_bg+b_bg):.4f}")
print(f"solo T >= {UMBRAL} dias (n={len(j_vet):,}):   a={a_v:.4f}  b={b_v:.4f}  E[p]={a_v/(a_v+b_v):.4f}")
print(f"\n% clase cero  toda la muestra: {100*(x_obs==0).mean():.1f}%"
      f"   |  veteranos: {100*(x_obs[vet]==0).mean():.1f}%")

$E[p]$ pasa de ~0,27 a ~0,03: **un orden de magnitud** según a quién se le pregunte. Ese
salto tiene dos causas mezcladas, y conviene no confundirlas:

1. **Censura.** El ajuste supone que todos murieron tras su última compra. Entre los
   clientes recién captados hay muchos vivos-pero-callados contados como muertos, y eso
   infla $p$.
2. **Selección.** Los `T` altos **no son una cohorte de captación**: son el borde de
   diciembre de 2022, o sea todo el que ya era cliente cuando empiezan los datos. Están
   sesgados hacia los fieles por construcción — su clase cero es 5,4 % contra 30,3 % del total.

Las dos empujan en la misma dirección, así que este contraste **no mide la censura sola**.
Lo que sí deja claro: **ningún número único de abandono describe esta base**, y cualquier
«tasa de fuga» global va a depender del recorte más que del negocio.

Separar las dos causas es justamente lo que hace el BG/NBD completo — usa $t_x$ y $T$ a la
vez para decidir, cliente a cliente, si el silencio es muerte o lentitud.

---

## 6 · Lo que la EDA deja listo

**Para el modelo:**

- `tabla` con `sk_cliente`, `x`, `t_x`, `T`, `monetario` — las cinco invariantes pasan.
- El NBD ajustado da $(r, \alpha)$ y el BG suelto da $(a, b)$: **cuatro números que sirven
  de punto de partida** para el ajuste conjunto del BG/NBD, en vez de arrancar en
  $(1,1,1,1)$ a ciegas.

**Para entender la fuga:**

- **La población es extrema.** Media 20,4 recompras contra mediana 3, y `Var/media ≈ 111`.
  El «cliente promedio» no existe: cualquier número agregado sobre esta base engaña.
  El $r = 0{,}31 < 1$ del ajuste lo confirma — Gamma en forma de «L».
- **El 30,3 % es clase cero.** Decide la elección de modelo: BG/NBD asigna
  $P(\text{vivo}) = 1$ a *todos* los clientes con $x = 0$, por construcción. Con 30 % es
  incómodo pero manejable; por encima del 50 % habría que ir directo a MBG/NBD. Y es
  justo donde el BG suelto falla (predice 13,3 mil frente a 15,2 mil observados), lo que
  ya inclina la balanza hacia el MBG/NBD.
- **La retención sube con cada recompra** (0,70 → 0,97). No es que los clientes se
  vuelvan fieles: es que la mezcla se depura sola, los volubles caen primero. Por eso una
  tasa de fuga global no describe a nadie — y por eso $E[p]$ se mueve de 0,27 a 0,03 solo
  con cambiar el recorte.
- **La cola de `x`** llega a 798 recompras en 29 meses: casi tres al día. Hay que mirar a
  los 20 más extremos antes de ajustar —revendedores, cuentas corporativas— porque la
  Gamma tuerce su forma para acomodarlos a costa de todos los demás.
- **El pico de `T`**: 28,6 % de los clientes tienen `T` en el último mes posible. No es
  una cohorte de captación, es el borde de diciembre de 2022. Cualquier lectura por
  antigüedad tiene que tenerlo en cuenta.
- **834 clientes con `monetario` negativo**: Gamma-Gamma no los admite, hay que excluirlos
  en la fase de CLV.

**Lo que esta EDA no puede contestar** — y para lo que existe el modelo completo:
separar al **muerto** del **lento**. Eso necesita `t_x` y `T` a la vez, que es
justamente lo que ni el NBD ni el BG usan por separado.

Siguiente paso: partir en calibración y holdout —recalculando `T` contra `FIN_CAL`,
no contra `CORTE`, o hay fuga de información— y ajustar los cuatro parámetros juntos.

In [ ]:
salida = RAIZ / "notebooks" / "rfm_muestra.parquet"
tabla.write_parquet(salida)
print(f"guardado: {salida}  ({len(tabla):,} clientes)")

print("\nparametros de arranque para el ajuste conjunto BG/NBD:")
print(f"  r={r_nbd:.4f}  alpha={alpha_nbd:.4f}  a={a_bg:.4f}  b={b_bg:.4f}")